In [1]:
# STEP 1: IMPORT NECESSARY LIBRARIES
import requests
import pandas as pd
import numpy as np
import re
import time

In [3]:
import requests
import pandas as pd

print("Fetching product data...")

all_products = []

# Fetch multiple pages
for i in range(1, 10):
    url = f"https://fakestoreapi.com/products?limit=20"

    res = requests.get(url)

    if res.status_code == 200:
        data = res.json()

        for p in data:
            all_products.append({
                "product_id": p.get("id"),
                "title": p.get("title"),
                "price": p.get("price"),
                "category": p.get("category")})

df_products = pd.DataFrame(all_products)

print("✅ Product Data Shape:", df_products.shape)
df_products.head()

Fetching product data...
✅ Product Data Shape: (180, 4)


,product_id,title,price,category
0,1,"Fjallraven - Foldsack No. 1 Backpack, Fits 15 ...",109.95,men's clothing
1,2,Mens Casual Premium Slim Fit T-Shirts,22.30,men's clothing
2,3,Mens Cotton Jacket,55.99,men's clothing
3,4,Mens Casual Slim Fit,15.99,men's clothing
4,5,John Hardy Women's Legends Naga Gold & Silver ...,695.00,jewelery


In [8]:
# STEP 3: FETCH TEXT DATA (STACK EXCHANGE)

print("\nFetching user text data...")
reviews = []

for page in range(1, 6):  # 5 pages = ~500 rows

    print(f"Fetching page {page}")

    text_url = "https://api.stackexchange.com/2.3/questions"
    params = {
        "order": "desc",
        "sort": "activity",
        "site": "stackoverflow",
        "pagesize": 100,
        "page": page}

    res = requests.get(text_url, params=params)
    data = res.json()

    questions = data.get("items", [])

    for q in questions:
        reviews.append(q.get("title", ""))

    time.sleep(1)

df_reviews = pd.DataFrame({
    "user_id": range(len(reviews)),
    "review_text": reviews})

print("✅ Text Data Shape:", df_reviews.shape)


Fetching user text data...
Fetching page 1
Fetching page 2
Fetching page 3
Fetching page 4
Fetching page 5
✅ Text Data Shape: (500, 2)


In [10]:
# Drop duplicates
df_reviews = df_reviews.drop_duplicates(subset=["review_text"])
df_products = df_products.drop_duplicates(subset=["product_id"])

In [12]:
# STEP 4: MERGE DATA

print("\nCombining datasets...")

# Randomly assign products to reviews
df_reviews['product_id'] = df_products['product_id'].sample(
    n=len(df_reviews), replace=True).values

# Merge
df_final = pd.merge(df_reviews, df_products, on='product_id', how='left')
df_final = df_final.drop_duplicates()

print("✅ Final Dataset Shape:", df_final.shape)
print(df_final.head())


Combining datasets...
✅ Final Dataset Shape: (500, 6)
   user_id                                        review_text  product_id  \
0        0  (SOLVED) Python Lexer Reading Numbers Function...          18   
1        1  Is it possible to &quot;gracefully&quot; stop ...           3   
2        2         Are Unicode and ASCII characters the same?          14   
3        3  Why does @XmlJavaTypeAdapter on the root objec...           5   
4        4    How to type a decorator as a callable on Python          20   

                                               title   price          category  
0        MBJ Women's Solid Short Sleeve Boat Neck V     9.85  women's clothing  
1                                 Mens Cotton Jacket   55.99    men's clothing  
2  Samsung 49-Inch CHG90 144Hz Curved Gaming Moni...  999.99       electronics  
3  John Hardy Women's Legends Naga Gold & Silver ...  695.00          jewelery  
4         DANVOUY Womens T Shirt Casual Cotton Short   12.99  women's clothin

In [13]:
# STEP 5: CLEAN DATASET

# Remove test products
df_final = df_final[~df_final['title'].str.contains("test", case=False, na=False)]
df_final = df_final[~df_final['title'].str.contains("UNIQPRODUCT", case=False, na=False)]
df_final['title'] = df_final['title'].replace(
    to_replace=r"Updated.*",
    value="Generic Product",
    regex=True)

# HANDLING PRICE (NOT DROPPING THE ROWS)
# Convert invalid prices to NaN
df_final['price'] = pd.to_numeric(df_final['price'], errors='coerce')

# Replace unrealistic values
df_final.loc[df_final['price'] <= 5, 'price'] = np.nan
df_final.loc[df_final['price'] >= 1000, 'price'] = np.nan

# Fill missing prices with median
df_final['price'] = df_final['price'].fillna(df_final['price'].median())

# Clean text for EDA analysis and sentiment analysis
def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)              # remove HTML tags
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)    # remove special chars
    text = re.sub(r'\s+', ' ', text).strip()      # remove extra spaces
    return text

df_final['cleaned_text'] = df_final['review_text'].apply(clean_text)

# Reset index
df_final.reset_index(drop=True, inplace=True)

print("\n✅ CLEANED DATASET SHAPE:", df_final.shape)
df_final.head()


✅ CLEANED DATASET SHAPE: (500, 7)


,user_id,review_text,product_id,title,price,category,cleaned_text
0,0,(SOLVED) Python Lexer Reading Numbers Function...,18,MBJ Women's Solid Short Sleeve Boat Neck V,9.85,women's clothing,solved python lexer reading numbers function p...
1,1,Is it possible to &quot;gracefully&quot; stop ...,3,Mens Cotton Jacket,55.99,men's clothing,is it possible to quotgracefullyquot stop a un...
2,2,Are Unicode and ASCII characters the same?,14,Samsung 49-Inch CHG90 144Hz Curved Gaming Moni...,999.99,electronics,are unicode and ascii characters the same
3,3,Why does @XmlJavaTypeAdapter on the root objec...,5,John Hardy Women's Legends Naga Gold & Silver ...,695.00,jewelery,why does xmljavatypeadapter on the root object...
4,4,How to type a decorator as a callable on Python,20,DANVOUY Womens T Shirt Casual Cotton Short,12.99,women's clothing,how to type a decorator as a callable on python


In [17]:
# STEP 6: SAVE

df_final.to_csv("final_dataset_fstoreAPI.csv", index=False)

print("\r Dataset saved as 'final_dataset_fstoreAPI.csv'")

 Dataset saved as 'final_dataset_fstoreAPI.csv'


In [15]:
# STEP 7: SAVE TO DRIVE
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

save_path = "/content/drive/MyDrive/Capstone-Data Analytics/data/final_dataset_fstoreAPI.csv"

df_final.to_csv(save_path, index=False)

Mounted at /content/drive
